### Procesamiento de archivo individual

In [13]:
import re
import datetime
import pandas as pd

def extraer_precipitacion_dga(archivo_entrada, archivo_salida=None):
    """
    Extrae datos de precipitación diaria desde archivos Excel con formato DGA
    y los estructura en una serie temporal continua de 2 columnas: Fecha y Precipitacion.
    
    Parámetros:
    -----------
    archivo_entrada : str
        Ruta del archivo .xls o .xlsx a procesar.
    archivo_salida : str, opcional
        Ruta donde se guardará el archivo .csv generado.
        
    Retorna:
    --------
    pd.DataFrame
        DataFrame con columnas ['Fecha', 'Precipitacion'].
    """
    # Mapeo de abreviaturas de meses a número (incluyendo variantes como SET/SEP)
    meses_map = {
        'ENE': 1, 'FEB': 2, 'MAR': 3, 'ABR': 4,
        'MAY': 5, 'JUN': 6, 'JUL': 7, 'AGO': 8,
        'SEP': 9, 'OCT': 10, 'NOV': 11, 'DIC': 12
    }
    
    df_raw = pd.read_excel(archivo_entrada, sheet_name=0, header=None)
    registros = []
    
    i = 0
    total_filas = len(df_raw)
    
    while i < total_filas:
        fila = df_raw.iloc[i]
        fila_texto = " ".join([str(val) for val in fila if pd.notna(val)])
        
        # 1. Detectar el inicio de un bloque anual (ej. "AÑO 2000" o "ANO 2000")
        coincidencia_ano = re.search(r'A[ÑN]O\s*(\d{4})', fila_texto, re.IGNORECASE)
        
        if coincidencia_ano:
            ano_actual = int(coincidencia_ano.group(1))
            i += 1  # Fila de cabecera de meses (DIA, ENE, FEB, ...)
            
            if i < total_filas:
                fila_cabecera = df_raw.iloc[i]
                col_meses = {}
                col_dia = None
                
                # Identificar dinámicamente los índices de columna para cada mes y para el día
                for col_idx, val in enumerate(fila_cabecera):
                    if pd.isna(val):
                        continue
                    val_str = str(val).strip().upper()
                    if val_str == 'DIA':
                        col_dia = col_idx
                    elif val_str in meses_map:
                        col_meses[col_idx] = meses_map[val_str]
                
                # 2. Iterar por las filas de días (1 al 31)
                i += 1
                while i < total_filas:
                    r = df_raw.iloc[i]
                    if col_dia is not None and pd.notna(r[col_dia]):
                        try:
                            dia_num = int(float(str(r[col_dia]).strip()))
                            if 1 <= dia_num <= 31:
                                for col_idx, mes_num in col_meses.items():
                                    val_precip = r[col_idx]
                                    # Validar que la fecha exista en el calendario
                                    try:
                                        fecha_obj = datetime.date(ano_actual, mes_num, dia_num)
                                        # Convertir el valor a numérico; datos faltantes/texto quedan como NaN
                                        valor = pd.to_numeric(val_precip, errors='coerce') if pd.notna(val_precip) else None
                                        registros.append({
                                            'Fecha': fecha_obj,
                                            'Precipitacion': valor
                                        })
                                    except ValueError:
                                        # Días inválidos en el calendario se ignoran automáticamente
                                        pass
                                i += 1
                                continue
                        except (ValueError, TypeError):
                            pass
                    # Fin del bloque de días del año actual
                    break
                continue
        i += 1
        
    df_resultado = pd.DataFrame(registros)
    
    if not df_resultado.empty:
        # Ordenar cronológicamente y estandarizar formato YYYY-MM-DD
        df_resultado = df_resultado.sort_values('Fecha').reset_index(drop=True)
        df_resultado['Fecha'] = df_resultado['Fecha'].astype(str)
        
        if archivo_salida:
            df_resultado.to_csv(archivo_salida, index=False)
            
    return df_resultado


In [ ]:
# Procesa un archivo y guarda el resultado en CSV
df = extraer_precipitacion_dga(
    archivo_entrada='../data/precipitaciones-cuenca/dga/2000-2003_coyacagua.xls',
    archivo_salida='../outputs/dga_precipitation/2000-2003_coyacagua.csv'
)
print(df.head())

        Fecha  Precipitacion
0  2000-01-01            0.0
1  2000-01-02            0.0
2  2000-01-03            0.0
3  2000-01-04            0.0
4  2000-01-05            0.0


### Procesamiento de múltiples archivos

In [15]:
import os
import glob
import re
import datetime
import unicodedata
import pandas as pd


def _sanitizar_nombre_estacion(nombre: str) -> str:
    """
    Limpia el nombre de la estación para usarlo en nombres de columnas:
    elimina acentos, pasa a minúsculas y reemplaza espacios/guiones por guiones bajos.
    Ejemplo: 'ALTOS DEL HUASCO' -> 'altos_del_huasco'
    """
    # Normalizar para remover tildes y caracteres especiales
    nombre = unicodedata.normalize('NFKD', str(nombre)).encode('ASCII', 'ignore').decode('utf-8')
    nombre = re.sub(r'[\s\-]+', '_', nombre.strip().lower())
    nombre = re.sub(r'[^a-z0-9_]', '', nombre)
    return re.sub(r'_+', '_', nombre).strip('_')


def _extraer_datos_hoja(df_raw: pd.DataFrame) -> list:
    """
    Parsea la matriz de datos de una sola hoja Excel en formato DGA.
    Retorna una lista de tuplas/diccionarios (fecha_str, valor_precip).
    """
    meses_map = {
        'ENE': 1, 'FEB': 2, 'MAR': 3, 'ABR': 4,
        'MAY': 5, 'JUN': 6, 'JUL': 7, 'AGO': 8,
        'SEP': 9, 'SET': 9, 'OCT': 10, 'NOV': 11, 'DIC': 12
    }
    
    registros = []
    i = 0
    total_filas = len(df_raw)
    
    while i < total_filas:
        fila = df_raw.iloc[i]
        fila_texto = " ".join([str(val) for val in fila if pd.notna(val)])
        
        # Detectar el bloque de inicio de un año (ej. "AÑO 2012")
        coincidencia_ano = re.search(r'A[ÑN]O\s*(\d{4})', fila_texto, re.IGNORECASE)
        
        if coincidencia_ano:
            ano_actual = int(coincidencia_ano.group(1))
            i += 1  # Cabecera de meses (DIA, ENE, FEB, ...)
            
            if i < total_filas:
                fila_cabecera = df_raw.iloc[i]
                col_meses = {}
                col_dia = None
                
                for col_idx, val in enumerate(fila_cabecera):
                    if pd.isna(val):
                        continue
                    val_str = str(val).strip().upper()
                    if val_str == 'DIA':
                        col_dia = col_idx
                    elif val_str in meses_map:
                        col_meses[col_idx] = meses_map[val_str]
                
                # Leer filas de días (1 al 31)
                i += 1
                while i < total_filas:
                    r = df_raw.iloc[i]
                    if col_dia is not None and pd.notna(r[col_dia]):
                        try:
                            dia_num = int(float(str(r[col_dia]).strip()))
                            if 1 <= dia_num <= 31:
                                for col_idx, mes_num in col_meses.items():
                                    val_precip = r[col_idx]
                                    try:
                                        fecha_obj = datetime.date(ano_actual, mes_num, dia_num)
                                        valor = pd.to_numeric(val_precip, errors='coerce') if pd.notna(val_precip) else None
                                        registros.append({
                                            'date': str(fecha_obj),
                                            'precip': valor
                                        })
                                    except ValueError:
                                        # Descarta días inexistentes (ej: 30 de febrero)
                                        pass
                                i += 1
                                continue
                        except (ValueError, TypeError):
                            pass
                    break
                continue
        i += 1
        
    return registros


def consolidar_precipitacion_dga(ruta_entrada: str, archivo_salida_csv: str = None) -> pd.DataFrame:
    """
    Procesa uno o múltiples archivos Excel (.xls / .xlsx) con formato DGA,
    recorriendo todas las hojas/estaciones y consolidando los datos en formato ancho.
    
    Parámetros:
    -----------
    ruta_entrada : str
        Ruta a una carpeta que contiene los archivos .xls/.xlsx, o ruta a un archivo individual.
    archivo_salida_csv : str, opcional
        Ruta donde se exportará el archivo CSV generado.
        
    Retorna:
    --------
    pd.DataFrame
        DataFrame en formato ancho con columnas: ['date', 'estacion1_precip_mm', 'estacion2_precip_mm', ...]
    """
    # Identificar si la entrada es un directorio o un archivo específico
    if os.path.isdir(ruta_entrada):
        archivos = glob.glob(os.path.join(ruta_entrada, "*.xls")) + glob.glob(os.path.join(ruta_entrada, "*.xlsx"))
    elif os.path.isfile(ruta_entrada):
        archivos = [ruta_entrada]
    else:
        archivos = glob.glob(ruta_entrada)
        
    archivos = sorted(list(set(archivos)))
    
    if not archivos:
        print(f"No se encontraron archivos .xls o .xlsx en: {ruta_entrada}")
        return pd.DataFrame()
        
    todos_los_registros = []
    
    for arch in archivos:
        try:
            excel = pd.ExcelFile(arch)
            for nombre_hoja in excel.sheet_names:
                df_hoja = pd.read_excel(arch, sheet_name=nombre_hoja, header=None)
                registros = _extraer_datos_hoja(df_hoja)
                
                col_estacion = f"{_sanitizar_nombre_estacion(nombre_hoja)}_precip_mm"
                
                for r in registros:
                    todos_los_registros.append({
                        'date': r['date'],
                        'station_col': col_estacion,
                        'precip': r['precip']
                    })
        except Exception as e:
            print(f"Error procesando el archivo '{arch}': {e}")
            
    if not todos_los_registros:
        print("No se encontraron registros válidos de precipitación.")
        return pd.DataFrame()
        
    df_long = pd.DataFrame(todos_los_registros)
    
    # Pivotar a formato ancho preservando todos los registros de fechas
    df_wide = df_long.pivot_table(
        index='date',
        columns='station_col',
        values='precip',
        aggfunc='first',
        dropna=False
    ).reset_index()
    
    # Ordenar cronológicamente
    df_wide['date_dt'] = pd.to_datetime(df_wide['date'])
    df_wide = df_wide.sort_values('date_dt').drop(columns=['date_dt']).reset_index(drop=True)
    df_wide.columns.name = None
    
    # Guardar en CSV si se especificó ruta de salida
    if archivo_salida_csv:
        df_wide.to_csv(archivo_salida_csv, index=False)
        print(f"Archivo exportado exitosamente en: {archivo_salida_csv}")
        
    return df_wide


In [ ]:
# Lee todos los .xls y .xlsx dentro de un directorio
df_consolidado = consolidar_precipitacion_dga(
    ruta_entrada="../data/precipitaciones-cuenca/dga/",
    archivo_salida_csv="../outputs/dga_precipitation/dga_daily_precip.csv"
)

print(df_consolidado.head())


Archivo exportado exitosamente en: ../outputs/precipitation/dga_daily_precip.csv
         date  altos_del_huasco_precip_mm  coyacagua_precip_mm  \
0  2000-01-01                         NaN                  0.0   
1  2000-01-02                         NaN                  0.0   
2  2000-01-03                         NaN                  0.0   
3  2000-01-04                         NaN                  0.0   
4  2000-01-05                         NaN                  0.0   

   diablo_marca_precip_mm  rio_piga_en_collacagua_precip_mm  \
0                     NaN                               NaN   
1                     NaN                               NaN   
2                     NaN                               NaN   
3                     NaN                               NaN   
4                     NaN                               NaN   

   sillillica_precip_mm  
0                   NaN  
1                   NaN  
2                   NaN  
3                   NaN  
4            